In [0]:
# Notebook: save_raw_data
# Descrição: Salva dados brutos na camada Bronze

import requests
from datetime import datetime

url = "https://api.open-meteo.com/v1/forecast"

cities = {
    "Sao_Paulo": (-23.55, -46.63),
    "Rio_de_Janeiro": (-22.90, -43.20),
    "Brasilia": (-15.79, -47.88),
    "New_York": (40.71, -74.00),
    "London": (51.50, -0.12),
    "Tokyo": (35.68, 139.65),
    "Paris": (48.85, 2.35),
    "Berlin": (52.52, 13.40),
    "Sydney": (-33.86, 151.20),
    "Dubai": (25.20, 55.27)
}

rows = []

for city, coords in cities.items():
    params = {
        "latitude": coords[0],
        "longitude": coords[1],
        "current_weather": True
    }
    
    r = requests.get(url, params=params).json()
    weather = r["current_weather"]
    
    rows.append({
        "city": city,
        "latitude": coords[0],
        "longitude": coords[1],
        "temperature": weather["temperature"],
        "windspeed": weather["windspeed"],
        "winddirection": weather["winddirection"],
        "weathercode": weather["weathercode"],
        "time": weather["time"],
        "ingestion_timestamp": datetime.now().isoformat()
    })

df = spark.createDataFrame(rows)
print(f" Coletados {len(rows)} registros")

df.write.format("delta") \
  .mode("append") \
  .option("mergeSchema", "true") \
  .saveAsTable("weather_bronze")

print(" Dados salvos na tabela weather_bronze")

bronze = spark.table("weather_bronze")

In [0]:
print(f"Total de registros na Bronze: {bronze.count()}")
display(bronze.limit(10))